# FRED — Economic & Ag-Input Indicators (Federal Reserve)

**What it is.** The St. Louis Fed's economic database — hundreds of thousands of time series.
For Tracera: **input costs** (fuel, chemicals PPI), farm-product prices, interest rates, and
macro context (CPI/PPI).

| | |
|---|---|
| **Coverage** | US macro + sector series, long histories |
| **Cost / key** | **Free · API key required** |
| **Sign up** | https://fredaccount.stlouisfed.org/apikeys → `.env` as `FRED_API_KEY` |
| **Docs** | https://fred.stlouisfed.org/docs/api/fred/ |

**Why it's in Tracera.** Margins depend on input costs and rates, not just crop prices. FRED is
the free backbone for the cost/macro side of a farm P&L.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Setup — key guard

In [2]:
FRED_KEY = get_key("FRED_API_KEY", required=False)
print("FRED_API_KEY present:", bool(FRED_KEY))
if not FRED_KEY:
    print("→ Add FRED_API_KEY to .env to run this. Free key: https://fredaccount.stlouisfed.org/apikeys")

FRED_API_KEY present: True


### 2. Core query — pull a basket of ag-relevant series

In [3]:
FRED = "https://api.stlouisfed.org/fred/series/observations"

def fred_series(series_id, start="2015-01-01"):
    r = requests.get(FRED, params={"series_id": series_id, "api_key": FRED_KEY,
        "file_type": "json", "observation_start": start}, timeout=60)
    r.raise_for_status()
    obs = pd.DataFrame(r.json()["observations"])
    obs["date"] = pd.to_datetime(obs["date"])
    obs["value"] = pd.to_numeric(obs["value"], errors="coerce")
    return obs.set_index("date")["value"].rename(series_id)

# Verified series IDs (search https://fred.stlouisfed.org for more specific ones):
SERIES = {
    "PPIACO":       "PPI: all commodities",
    "WPU012":       "PPI: farm products - grain",
    "WPU0652":      "PPI: chemicals & allied products",
    "DCOILWTICO":   "WTI crude oil ($/bbl)",
    "MORTGAGE30US": "30-yr mortgage rate (%)",
}
if FRED_KEY:
    econ = pd.concat([fred_series(sid) for sid in SERIES], axis=1, sort=True)
    econ.columns = list(SERIES.values())
    print("Annual-average ag/econ indicators:")
    display(econ.resample("YE").mean().round(1).tail())
else:
    print("Skipped live call — add FRED_API_KEY. Series basket is defined above and ready.")

Annual-average ag/econ indicators:


,PPI: all commodities,PPI: farm products - grain,PPI: chemicals & allied products,WTI crude oil ($/bbl),30-yr mortgage rate (%)
date,,,,,
2022-12-31,264.5,287.3,464.2,94.9,5.3
2023-12-31,255.7,232.8,321.7,77.6,6.8
2024-12-31,254.6,171.6,307.6,76.6,6.7
2025-12-31,260.3,168.0,347.7,65.4,6.6
2026-12-31,276.9,169.1,397.1,83.9,6.3


### Notes & how to extend
- **Find series:** search https://fred.stlouisfed.org or the `fred/series/search` API endpoint;
  every series has a stable ID you drop into `fred_series()`. E.g. fertilizer-specific PPIs live
  under `WPU0652013*`.
- **Useful themes:** input PPIs (fuel, fertilizer, chemicals), farm real-estate value, ag
  interest rates, exchange rates (export competitiveness), diesel prices.
- **Convenience:** the `fredapi` package wraps this (`Fred(api_key=...).get_series(id)`); the
  raw REST above avoids the extra dependency.